# SAP ECC Treasury Data — Cleaning Pipeline
**Input:** Raw SAP ECC treasury export (`sap_ecc_treasury_export_raw.csv`) — 10,300+ rows with known data quality issues.  
**Output:** Clean, validated dataset ready for encoding and AI analysis.  
**Approach:** Systematic cleaning in priority order — structural issues first, then type corrections, then business logic validation.

## 1. Load & Initial Assessment
First step in any data pipeline: understand what you're working with before touching anything.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import warnings
warnings.filterwarnings('ignore')

# Load the raw export
df_raw = pd.read_csv('sap_ecc_treasury_export_raw.csv', encoding='utf-8-sig', dtype=str)

# Keep a copy for comparison at the end
original_row_count = len(df_raw)

# Working copy
df = df_raw.copy()

print(f'Loaded {len(df)} rows, {len(df.columns)} columns')
print(f'\nColumn types (all loaded as string intentionally):')
print(df.dtypes.value_counts())
print(f'\nMemory usage: {df.memory_usage(deep=True).sum() / (1024*1024):.1f} MB')

Loaded 10300 rows, 34 columns

Column types (all loaded as string intentionally):
object    34
Name: count, dtype: int64

Memory usage: 18.2 MB


In [2]:
# Data quality snapshot BEFORE cleaning
print('=== DATA QUALITY SNAPSHOT (BEFORE) ===')
print(f'\nTotal rows: {len(df)}')
print(f'Total columns: {len(df.columns)}')

# Null/empty counts per column
null_report = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'empty_string': (df == '').sum(),
    'total_missing': df.isnull().sum() + (df == '').sum(),
    'pct_missing': ((df.isnull().sum() + (df == '').sum()) / len(df) * 100).round(2)
})
print(f'\nColumns with missing data:')
print(null_report[null_report['total_missing'] > 0].sort_values('total_missing', ascending=False))

# Duplicate check
dup_count = df.duplicated(subset=['BELNR', 'BUZEI', 'BUKRS', 'GJAHR']).sum()
print(f'\nDuplicate rows (by BELNR+BUZEI+BUKRS+GJAHR): {dup_count}')

=== DATA QUALITY SNAPSHOT (BEFORE) ===

Total rows: 10300
Total columns: 34

Columns with missing data:
                   null_count  empty_string  total_missing  pct_missing
MATURITY_DATE            7704             0           7704        74.80
KUNNR                    7651             0           7651        74.28
COUNTERPARTY_BANK        6128             0           6128        59.50
LIFNR                    4814             0           4814        46.74
ZUESSION                 3182             0           3182        30.89
NAME1                     800             0            800         7.77
VALUT                     495             0            495         4.81
SGTXT                     391             0            391         3.80
KURSF                     108             0            108         1.05
WAERS                     100             0            100         0.97

Duplicate rows (by BELNR+BUZEI+BUKRS+GJAHR): 300


## 2. Remove Duplicates
SAP document numbers (BELNR) combined with line item (BUZEI), company code (BUKRS), and fiscal year (GJAHR) must be unique.  
Duplicates arise from overlapping report runs or repeated data extractions.

In [3]:
before = len(df)

# Deduplicate on the SAP primary key: document number + line item + company code + fiscal year
df = df.drop_duplicates(subset=['BELNR', 'BUZEI', 'BUKRS', 'GJAHR'], keep='first')
df = df.reset_index(drop=True)

after = len(df)
print(f'Removed {before - after} duplicate rows')
print(f'Rows: {before} → {after}')

Removed 300 duplicate rows
Rows: 10300 → 10000


## 3. Standardize Date Formats
The raw export contains mixed date formats: `YYYY-MM-DD`, `DD.MM.YYYY` (German SAP), and `MM/DD/YYYY` (US).  
All dates must be normalized to `YYYY-MM-DD` (ISO 8601) for consistent processing.

In [4]:
date_columns = ['BUDAT', 'BLDAT', 'VALUT', 'CPUDT', 'AEDAT', 'MATURITY_DATE']

def parse_mixed_date(val):
    """
    Parse dates that could be in any of three formats:
    - YYYY-MM-DD (ISO)
    - DD.MM.YYYY (German/SAP)
    - MM/DD/YYYY (US)
    Returns standardized YYYY-MM-DD string or NaN.
    """
    if pd.isna(val) or val == '' or val == 'nan':
        return np.nan
    
    val = str(val).strip()
    
    # Try each format in order of likelihood
    for fmt in ['%Y-%m-%d', '%d.%m.%Y', '%m/%d/%Y']:
        try:
            return datetime.strptime(val, fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue
    
    return np.nan  # Unparseable date

# Apply to all date columns
for col in date_columns:
    before_nulls = df[col].isna().sum() + (df[col] == '').sum()
    df[col] = df[col].apply(parse_mixed_date)
    after_nulls = df[col].isna().sum()
    parsed = before_nulls - after_nulls if after_nulls < before_nulls else 0
    print(f'{col}: standardized | nulls before={before_nulls}, after={after_nulls}')

# Verify: show sample of each date column
print(f'\nSample dates after standardization:')
print(df[date_columns].head(10))

BUDAT: standardized | nulls before=0, after=0
BLDAT: standardized | nulls before=0, after=0
VALUT: standardized | nulls before=489, after=489
CPUDT: standardized | nulls before=0, after=0
AEDAT: standardized | nulls before=0, after=0
MATURITY_DATE: standardized | nulls before=7471, after=7471

Sample dates after standardization:
        BUDAT       BLDAT       VALUT       CPUDT       AEDAT MATURITY_DATE
0  2024-09-04  2024-09-02  2024-09-07  2024-09-05  2024-09-05           NaN
1  2024-10-22  2024-10-18  2024-10-25  2024-10-24  2024-10-24           NaN
2  2024-02-24  2024-02-22  2024-02-27  2024-02-25  2024-02-25           NaN
3  2024-05-13  2024-05-13  2024-05-14  2024-05-15  2024-05-15           NaN
4  2024-05-25  2024-05-25  2024-05-26  2024-05-27  2024-05-27    2024-06-24
5  2024-07-19  2024-07-16  2024-07-21  2024-07-20  2024-07-20           NaN
6  2024-07-01  2024-07-01  2024-07-03  2024-07-01  2024-07-01           NaN
7  2024-04-01  2024-03-31  2024-04-02  2024-04-01  2024-04-01

## 4. Fix Numeric Columns
Amounts (`WRBTR`, `DMBTR`), exchange rates (`KURSF`), and line items (`BUZEI`) must be proper numeric types.  
Handle negative amounts (sign errors), zero/null exchange rates on foreign currency rows.

In [5]:
# Convert amount columns to float
for col in ['WRBTR', 'DMBTR', 'KURSF']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert BUZEI to integer
df['BUZEI'] = pd.to_numeric(df['BUZEI'], errors='coerce').astype('Int64')

# --- Fix negative amounts (sign errors) ---
neg_wrbtr = (df['WRBTR'] < 0).sum()
df['WRBTR'] = df['WRBTR'].abs()
print(f'Fixed {neg_wrbtr} negative WRBTR values (converted to absolute)')

# --- Fix zero/null exchange rates on foreign currency transactions ---
# If WAERS != EUR and KURSF is 0 or NaN, recalculate from DMBTR/WRBTR
bad_rates = df[(df['WAERS'] != 'EUR') & (df['WAERS'] != '') & 
               ((df['KURSF'].isna()) | (df['KURSF'] == 0))]

print(f'Found {len(bad_rates)} foreign currency rows with missing/zero exchange rate')

# Recalculate rate where possible (DMBTR / WRBTR)
for idx in bad_rates.index:
    wrbtr = df.at[idx, 'WRBTR']
    dmbtr = df.at[idx, 'DMBTR']
    if pd.notna(wrbtr) and pd.notna(dmbtr) and wrbtr > 0:
        df.at[idx, 'KURSF'] = round(dmbtr / wrbtr, 5)
    else:
        # Fall back to standard rate lookup
        currency = df.at[idx, 'WAERS']
        fallback_rates = {'USD': 0.92, 'CHF': 1.04, 'GBP': 1.17, 'JPY': 0.0061}
        df.at[idx, 'KURSF'] = fallback_rates.get(currency, 1.0)

remaining_bad = df[(df['WAERS'] != 'EUR') & (df['WAERS'] != '') & 
                   ((df['KURSF'].isna()) | (df['KURSF'] == 0))].shape[0]
print(f'After fix: {remaining_bad} rows still have bad exchange rates')

print(f'\nNumeric column stats:')
print(df[['WRBTR', 'DMBTR', 'KURSF']].describe().round(2))

Fixed 98 negative WRBTR values (converted to absolute)
Found 197 foreign currency rows with missing/zero exchange rate
After fix: 0 rows still have bad exchange rates

Numeric column stats:
             WRBTR        DMBTR     KURSF
count     10000.00     10000.00  10000.00
mean     162959.33    159279.49      0.98
std      415694.52    406714.36      0.04
min          20.48        20.48      0.89
25%       10489.63     10206.06      0.94
50%       43288.38     42359.54      0.99
75%      154627.37    150968.86      1.01
max    11614323.62  10932911.25      1.07


## 5. Fix Currency Issues
Some rows have blank `WAERS` (currency code) despite having amounts filled.  
Also fix rows where `DMBTR == WRBTR` but `WAERS != EUR` (missing currency conversion).

In [6]:
# --- Fix blank WAERS ---
blank_waers = df[(df['WAERS'] == '') | (df['WAERS'].isna())]
print(f'Rows with blank WAERS: {len(blank_waers)}')

# Strategy: infer from vendor/customer master data, or default to EUR (local currency)
vendor_currency = {k: v['currency'] for k, v in {
    'V001': {'currency': 'USD'}, 'V002': {'currency': 'EUR'},
    'V003': {'currency': 'USD'}, 'V004': {'currency': 'EUR'},
    'V005': {'currency': 'USD'}, 'V006': {'currency': 'EUR'},
    'V007': {'currency': 'USD'}, 'V008': {'currency': 'EUR'},
    'V009': {'currency': 'USD'}, 'V010': {'currency': 'USD'}
}.items()}

customer_currency = {k: v['currency'] for k, v in {
    'C001': {'currency': 'EUR'}, 'C002': {'currency': 'EUR'},
    'C003': {'currency': 'EUR'}, 'C004': {'currency': 'EUR'},
    'C005': {'currency': 'EUR'}, 'C006': {'currency': 'EUR'},
    'C007': {'currency': 'EUR'}, 'C008': {'currency': 'CHF'}
}.items()}

fixed_waers = 0
for idx in blank_waers.index:
    lifnr = df.at[idx, 'LIFNR']
    kunnr = df.at[idx, 'KUNNR']
    if lifnr in vendor_currency:
        df.at[idx, 'WAERS'] = vendor_currency[lifnr]
        fixed_waers += 1
    elif kunnr in customer_currency:
        df.at[idx, 'WAERS'] = customer_currency[kunnr]
        fixed_waers += 1
    else:
        df.at[idx, 'WAERS'] = 'EUR'  # Default to local currency
        fixed_waers += 1

print(f'Fixed {fixed_waers} blank WAERS values')

# --- Fix DMBTR == WRBTR on non-EUR rows (missing conversion) ---
bad_conversion = df[(df['WAERS'] != 'EUR') & 
                    (df['WRBTR'] == df['DMBTR']) & 
                    (df['WRBTR'] > 0)]
print(f'\nRows with missing currency conversion (DMBTR == WRBTR, non-EUR): {len(bad_conversion)}')

for idx in bad_conversion.index:
    rate = df.at[idx, 'KURSF']
    if pd.notna(rate) and rate > 0:
        df.at[idx, 'DMBTR'] = round(df.at[idx, 'WRBTR'] * rate, 2)

remaining = df[(df['WAERS'] != 'EUR') & 
               (df['WRBTR'] == df['DMBTR']) & 
               (df['WRBTR'] > 0)].shape[0]
print(f'After fix: {remaining} rows still have unconverted amounts')

Rows with blank WAERS: 100
Fixed 100 blank WAERS values

Rows with missing currency conversion (DMBTR == WRBTR, non-EUR): 77
After fix: 6 rows still have unconverted amounts


## 6. Clean Text Fields
Fix `SGTXT` (item text): trim whitespace, normalize casing, handle special characters (German umlauts), remove empty entries.  
Also fill missing `NAME1` (counterparty name) from vendor/customer master data where possible.

In [7]:
# --- Clean SGTXT ---
before_empty = (df['SGTXT'] == '').sum() + df['SGTXT'].isna().sum()

# Strip whitespace
df['SGTXT'] = df['SGTXT'].str.strip()

# Normalize casing to Title Case
df['SGTXT'] = df['SGTXT'].apply(
    lambda x: x.title() if pd.notna(x) and x != '' else x
)

# Fix common German umlaut replacements back to clean text
umlaut_map = {'ä': 'a', 'ö': 'o', 'ü': 'u', 'Ä': 'A', 'Ö': 'O', 'Ü': 'U'}
for old, new in umlaut_map.items():
    df['SGTXT'] = df['SGTXT'].str.replace(old, new, regex=False)

# Mark empty SGTXT as 'No Description'
df.loc[(df['SGTXT'] == '') | (df['SGTXT'].isna()), 'SGTXT'] = 'No Description'

after_empty = (df['SGTXT'] == '').sum() + (df['SGTXT'] == 'No Description').sum()
print(f'SGTXT: cleaned whitespace/casing/umlauts, marked {after_empty} empty entries as "No Description"')

# --- Fill missing NAME1 from master data ---
vendor_names = {
    'V001': 'Cisco Systems International', 'V002': 'Dell Technologies GmbH',
    'V003': 'Hewlett Packard Enterprise', 'V004': 'Microsoft Ireland Operations',
    'V005': 'Lenovo Global Technology', 'V006': 'VMware International Unlimited',
    'V007': 'Palo Alto Networks', 'V008': 'SAP SE',
    'V009': 'Juniper Networks', 'V010': 'Fortinet Inc.'
}
customer_names = {
    'C001': 'Bechtle AG', 'C002': 'CANCOM SE',
    'C003': 'SoftwareOne Deutschland', 'C004': 'Computacenter AG',
    'C005': 'Axians Deutschland', 'C006': 'T-Systems International',
    'C007': 'Atos IT Solutions', 'C008': 'Swisscom IT Services'
}

missing_name = (df['NAME1'] == '') | (df['NAME1'].isna())
print(f'\nNAME1 missing: {missing_name.sum()} rows')

filled = 0
for idx in df[missing_name].index:
    lifnr = df.at[idx, 'LIFNR']
    kunnr = df.at[idx, 'KUNNR']
    if lifnr in vendor_names:
        df.at[idx, 'NAME1'] = vendor_names[lifnr]
        filled += 1
    elif kunnr in customer_names:
        df.at[idx, 'NAME1'] = customer_names[kunnr]
        filled += 1
    else:
        df.at[idx, 'NAME1'] = 'Unknown Counterparty'
        filled += 1

print(f'Filled {filled} missing NAME1 values (from master data or marked Unknown)')

SGTXT: cleaned whitespace/casing/umlauts, marked 382 empty entries as "No Description"

NAME1 missing: 783 rows
Filled 783 missing NAME1 values (from master data or marked Unknown)


## 7. Date Validation & Business Logic
Apply SAP treasury business rules:  
- Posting date (`BUDAT`) must be within fiscal year 2024  
- Value date (`VALUT`) should not be before posting date  
- `DEAL_STATUS = SETTLED` should not have future `MATURITY_DATE`  
- `FX_FORWARD` deals must have a `MATURITY_DATE`  
- A row should have either `LIFNR` (vendor) or `KUNNR` (customer), not both

In [8]:
# --- Flag future posting dates ---
fiscal_year_end = '2024-12-31'
future_budat = df[df['BUDAT'] > fiscal_year_end]
print(f'Rows with BUDAT after fiscal year 2024: {len(future_budat)}')

# Cap future dates to fiscal year end
df.loc[df['BUDAT'] > fiscal_year_end, 'BUDAT'] = fiscal_year_end
print(f'→ Capped to {fiscal_year_end}')

# --- Fix VALUT < BUDAT ---
# Only compare where both dates exist
valid_dates = df[(df['VALUT'].notna()) & (df['BUDAT'].notna())]
valut_before_budat = valid_dates[valid_dates['VALUT'] < valid_dates['BUDAT']]
print(f'\nRows with VALUT < BUDAT: {len(valut_before_budat)}')

# Set VALUT = BUDAT where VALUT is earlier (conservative fix)
for idx in valut_before_budat.index:
    df.at[idx, 'VALUT'] = df.at[idx, 'BUDAT']
print(f'→ Set VALUT = BUDAT for {len(valut_before_budat)} rows')

# --- Fix SETTLED deals with future MATURITY_DATE ---
settled_future = df[(df['DEAL_STATUS'] == 'SETTLED') & 
                    (df['MATURITY_DATE'].notna()) & 
                    (df['MATURITY_DATE'] != '') &
                    (df['MATURITY_DATE'] > fiscal_year_end)]
print(f'\nSETTLED deals with future MATURITY_DATE: {len(settled_future)}')

# Change status to OPEN since maturity hasn't passed
df.loc[settled_future.index, 'DEAL_STATUS'] = 'OPEN'
print(f'→ Changed status to OPEN for {len(settled_future)} rows')

# --- Fix FX_FORWARD without MATURITY_DATE ---
fwd_no_maturity = df[(df['DEAL_TYPE'] == 'FX_FORWARD') & 
                     ((df['MATURITY_DATE'] == '') | (df['MATURITY_DATE'].isna()))]
print(f'\nFX_FORWARD with no MATURITY_DATE: {len(fwd_no_maturity)}')

# Set maturity to 90 days from posting date (standard forward tenor)
for idx in fwd_no_maturity.index:
    budat = df.at[idx, 'BUDAT']
    if pd.notna(budat):
        mat_date = (pd.to_datetime(budat) + pd.Timedelta(days=90)).strftime('%Y-%m-%d')
        df.at[idx, 'MATURITY_DATE'] = mat_date
print(f'→ Set MATURITY_DATE = BUDAT + 90 days for {len(fwd_no_maturity)} rows')

# --- Fix dual counterparty (both LIFNR and KUNNR filled) ---
dual_cp = df[(df['LIFNR'] != '') & (df['LIFNR'].notna()) & 
             (df['KUNNR'] != '') & (df['KUNNR'].notna())]
print(f'\nRows with both LIFNR and KUNNR: {len(dual_cp)}')

# Resolve based on document type: KZ/ZP = vendor, DZ = customer
for idx in dual_cp.index:
    blart = df.at[idx, 'BLART']
    if blart in ('KZ', 'ZP'):
        df.at[idx, 'KUNNR'] = ''  # Keep vendor, clear customer
    elif blart == 'DZ':
        df.at[idx, 'LIFNR'] = ''  # Keep customer, clear vendor
    else:
        df.at[idx, 'KUNNR'] = ''  # Default: keep vendor
print(f'→ Resolved based on document type')

Rows with BUDAT after fiscal year 2024: 50
→ Capped to 2024-12-31

Rows with VALUT < BUDAT: 142
→ Set VALUT = BUDAT for 142 rows

SETTLED deals with future MATURITY_DATE: 512
→ Changed status to OPEN for 512 rows

FX_FORWARD with no MATURITY_DATE: 20
→ Set MATURITY_DATE = BUDAT + 90 days for 20 rows

Rows with both LIFNR and KUNNR: 50
→ Resolved based on document type


## 8. Fill Remaining Nulls & Final Type Casting
Handle any remaining NaN values with appropriate defaults, and set proper column types for the clean dataset.

In [9]:
# --- Fill remaining VALUT nulls with BUDAT (best available estimate) ---
valut_still_null = df['VALUT'].isna().sum()
df['VALUT'] = df['VALUT'].fillna(df['BUDAT'])
print(f'Filled {valut_still_null} remaining VALUT nulls with BUDAT')

# --- Fill remaining KURSF nulls (should be minimal at this point) ---
kursf_still_null = df['KURSF'].isna().sum()
df['KURSF'] = df['KURSF'].fillna(1.0)  # Default to 1.0 (EUR to EUR)
print(f'Filled {kursf_still_null} remaining KURSF nulls with 1.0')

# --- Ensure string columns don't have NaN ---
string_cols = ['MANDT', 'BUKRS', 'BELNR', 'BLART', 'BSCHL', 'HKONT', 'SGTXT',
               'WAERS', 'LIFNR', 'KUNNR', 'NAME1', 'HBKID', 'HKTID', 'BANKN',
               'RZAWE', 'DEAL_TYPE', 'DEAL_ID', 'DEAL_STATUS', 'COUNTERPARTY_BANK',
               'PRCTR', 'KOSTL', 'ZUESSION', 'ERNAM']

for col in string_cols:
    if col in df.columns:
        df[col] = df[col].fillna('').astype(str)

# --- Round amounts to 2 decimal places ---
df['WRBTR'] = df['WRBTR'].round(2)
df['DMBTR'] = df['DMBTR'].round(2)
df['KURSF'] = df['KURSF'].round(5)

print(f'\nFinal null check:')
remaining_nulls = df.isnull().sum()
if remaining_nulls.sum() == 0:
    print('✓ No null values remaining')
else:
    print(remaining_nulls[remaining_nulls > 0])

Filled 489 remaining VALUT nulls with BUDAT
Filled 0 remaining KURSF nulls with 1.0

Final null check:
MATURITY_DATE    7451
dtype: int64


## 9. Cleaning Summary & Quality Report
Compare before vs after to document all transformations applied.

In [10]:
print('=' * 60)
print('CLEANING SUMMARY REPORT')
print('=' * 60)
print(f'\nRows: {original_row_count} → {len(df)} ({original_row_count - len(df)} removed)')
print(f'Columns: {len(df.columns)}')
print(f'\nTransformations applied:')
print(f'  1. Duplicate removal (BELNR+BUZEI+BUKRS+GJAHR key)')
print(f'  2. Date format standardization (3 formats → ISO 8601)')
print(f'  3. Negative amount correction (absolute values)')
print(f'  4. Exchange rate recovery (recalculated or fallback rates)')
print(f'  5. Currency code inference (from vendor/customer master)')
print(f'  6. Currency conversion fix (DMBTR recalculated)')
print(f'  7. Text normalization (whitespace, casing, umlauts)')
print(f'  8. NAME1 enrichment (from master data lookup)')
print(f'  9. Future posting dates capped to fiscal year end')
print(f'  10. VALUT < BUDAT correction')
print(f'  11. SETTLED + future maturity → status changed to OPEN')
print(f'  12. FX_FORWARD maturity date imputation (BUDAT + 90 days)')
print(f'  13. Dual counterparty resolution (doc type based)')
print(f'  14. Remaining nulls filled with appropriate defaults')
print(f'\nData quality score: {((1 - df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100):.2f}%')

# Quick distribution checks
print(f'\n--- Key Distribution Checks ---')
print(f'\nDEAL_STATUS:\n{df["DEAL_STATUS"].value_counts()}')
print(f'\nWAERS:\n{df["WAERS"].value_counts()}')
print(f'\nBLART:\n{df["BLART"].value_counts()}')

CLEANING SUMMARY REPORT

Rows: 10300 → 10000 (300 removed)
Columns: 34

Transformations applied:
  1. Duplicate removal (BELNR+BUZEI+BUKRS+GJAHR key)
  2. Date format standardization (3 formats → ISO 8601)
  3. Negative amount correction (absolute values)
  4. Exchange rate recovery (recalculated or fallback rates)
  5. Currency code inference (from vendor/customer master)
  6. Currency conversion fix (DMBTR recalculated)
  7. Text normalization (whitespace, casing, umlauts)
  8. NAME1 enrichment (from master data lookup)
  9. Future posting dates capped to fiscal year end
  10. VALUT < BUDAT correction
  11. SETTLED + future maturity → status changed to OPEN
  12. FX_FORWARD maturity date imputation (BUDAT + 90 days)
  13. Dual counterparty resolution (doc type based)
  14. Remaining nulls filled with appropriate defaults

Data quality score: 97.81%

--- Key Distribution Checks ---

DEAL_STATUS:
DEAL_STATUS
SETTLED      4497
OPEN         3587
PENDING      1407
CANCELLED     509
Name: 

## 10. Export Clean Dataset
Save the cleaned data for the next phase: encoding → AI analysis → decoding.

In [11]:
csv_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'sap_ecc_treasury_clean.csv')
xlsx_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'sap_ecc_treasury_clean.xlsx')

df.to_csv(csv_path, index=False, encoding='utf-8-sig')
df.to_excel(xlsx_path, index=False, engine='openpyxl')

print(f'Clean CSV saved: {csv_path}')
print(f'Clean Excel saved: {xlsx_path}')
print(f'\nCSV size: {os.path.getsize(csv_path) / (1024*1024):.1f} MB')
print(f'Excel size: {os.path.getsize(xlsx_path) / (1024*1024):.1f} MB')
print(f'\nPhase 2 complete. Ready for encoding and AI analysis.')

Clean CSV saved: c:\Users\bobur\Downloads\sap_ecc_treasury_clean.csv
Clean Excel saved: c:\Users\bobur\Downloads\sap_ecc_treasury_clean.xlsx

CSV size: 2.6 MB
Excel size: 1.8 MB

Phase 2 complete. Ready for encoding and AI analysis.
